# Atelier Préparation de Données Tabulaires

## Partie 1 – Explorer les données

### 1) Charger les données CSV

In [41]:
import pandas as pd
df = pd.read_csv("../data/smart_building_raw.csv")

### 2) Afficher les premières lignes du dataset

In [42]:
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


### 3) Afficher les dernières lignes du dataset

In [43]:
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


### 4) Combien d'observations contient le dataset ?

In [44]:
df.shape

(507, 14)

**Réponse.** Le dataset contient **507 observations** (lignes).

### 5) Combien de variables possède le dataset ?

In [45]:
df.shape[1]

14

**Réponse.** Le dataset possède **14 variables** (colonnes).

### 6) Identifier les variables numériques

In [46]:
df.dtypes

id_mesure               int64
date                      str
batiment                  str
type_batiment             str
zone                      str
temperature           float64
humidite              float64
co2                   float64
occupation            float64
consommation_kwh      float64
mode_climatisation        str
etat_systeme              str
jour_semaine              str
alerte                    str
dtype: object

In [47]:
# df.select_dtypes(include="number")
df.select_dtypes(include="number").columns

Index(['id_mesure', 'temperature', 'humidite', 'co2', 'occupation',
       'consommation_kwh'],
      dtype='str')

**Réponse.** Colonnes numériques *au sens du type* : `id_mesure`, `temperature`, `humidite`, `co2`, `occupation`, `consommation_kwh`.

Attention : `id_mesure` est un **identifiant** (il est fait de chiffres mais calculer sa moyenne n'a pas de sens). Les vraies variables numériques *exploitables comme mesures* sont donc les **5** autres : `temperature`, `humidite`, `co2`, `occupation`, `consommation_kwh`.

### 7) Identifier les variables catégorielles

In [48]:
df.select_dtypes(include = "str").columns

Index(['date', 'batiment', 'type_batiment', 'zone', 'mode_climatisation',
       'etat_systeme', 'jour_semaine', 'alerte'],
      dtype='str')

**Réponse.** Colonnes de type texte : `date`, `batiment`, `type_batiment`, `zone`, `mode_climatisation`, `etat_systeme`, `jour_semaine`, `alerte`.

On écarte `date`, qui n'est pas une catégorie mais une date stockée en texte (elle sera convertie à la tâche 8). Les **7** vraies variables catégorielles sont : `batiment`, `type_batiment`, `zone`, `mode_climatisation`, `etat_systeme`, `jour_semaine`, `alerte`.

> Note transversale : `select_dtypes` classe sur le **type de stockage**, pas sur le **sens** de la variable. `id_mesure` est stocké en nombre mais reste un identifiant ; `date` est stockée en texte mais reste une date.

### 8) Identifier les dates

**Première tentative (elle échoue volontairement).** La conversion directe :

```python
df["date"] = pd.to_datetime(df["date"])
```

lève l'erreur suivante :

```
ValueError: day 30 must be in range 1..28 for month 2 in year 2025
```

> **Ce que révèle l'erreur.** La conversion directe échoue avec `ValueError: day 30 must be in range 1..28 for month 2` : il existe des dates **impossibles** dans la colonne (par exemple un 30 février). C'est une **incohérence** (valeur physiquement invalide). Pandas refuse de deviner et s'arrête. On relance donc la conversion en neutralisant ces valeurs (cellule suivante).

In [49]:
# la conversion
df["date"] = pd.to_datetime(df["date"], errors="coerce")

In [50]:
# vérifier le type
df["date"].dtype

dtype('<M8[us]')

In [51]:
#compter les dates devenues manquantes
df["date"].isna().sum()

np.int64(5)

**Identification.** La seule variable de type date est `date`.

**Préparation (au-delà de la simple identification).** Contrairement aux variables numériques (tâche 6) et catégorielles (tâche 7), déjà reconnues telles quelles par pandas, une date lue depuis un CSV arrive stockée en **texte** (`str`) : pandas ne la "voit" pas comme une date. On la convertit donc avec `pd.to_datetime` pour établir son vrai type (`datetime64`, affiché ici `<M8[us]` par NumPy — `M8` = `datetime64`, `[us]` = précision à la microseconde).

La conversion a aussi révélé **5 dates impossibles** (ex. un 30 février), transformées en valeurs manquantes grâce à `errors="coerce"`. Cela anticipe la règle 12f de l'énoncé (une valeur manifestement erronée et non récupérable devient manquante) : la colonne `date` est donc déjà réglée pour les tâches 12-13. Une date manquante s'écrit `NaT` (*Not a Time*) : c'est, pour une colonne de dates, ce que `NaN` est pour une colonne de nombres.

**Intérêt de la conversion :** pouvoir extraire l'année / le mois / le jour / l'heure, trier chronologiquement, filtrer par période et calculer des durées — impossible tant que la colonne reste du texte.

### 9) Identifier les identifiants

In [52]:
df["id_mesure"].nunique()

500

**Réponse.** L'identifiant du dataset est `id_mesure` : c'est une étiquette qui numérote
chaque mesure (faite de chiffres, mais aucune moyenne n'a de sens dessus → ce n'est pas
une variable numérique exploitable, malgré son type `int64`).

`df["id_mesure"].nunique()` renvoie **500** pour **507** lignes : 7 numéros sont donc
répétés. `id_mesure` n'est pas parfaitement unique → signal de **doublons potentiels**,
à examiner à la tâche 14 (vrais doublons identiques, ou lignes distinctes partageant un
même id ?).

### 10) Statistiques : moyenne, médiane, min, max, écart-type et quartiles

In [53]:
df.describe()

,id_mesure,date,temperature,humidite,co2,occupation,consommation_kwh
count,507.000000,502,495.000000,496.000000,500.000000,501.000000,502.000000
mean,1251.114398,2025-03-04 13:26:03.346613,24.154141,57.864113,844.150000,44.850299,169.069323
min,1001.000000,2025-01-01 00:00:00,-30.000000,-12.000000,89.000000,-20.000000,-100.000000
25%,1125.500000,2025-02-01 01:30:00,21.600000,49.275000,623.750000,27.000000,136.875000
50%,1252.000000,2025-03-04 21:00:00,24.000000,57.550000,787.500000,46.000000,169.800000
75%,1376.500000,2025-04-04 22:30:00,26.000000,65.750000,952.000000,61.000000,202.975000
max,1500.000000,2025-05-05 18:00:00,96.000000,160.000000,6000.000000,116.000000,336.200000
std,144.782769,NaN,7.418465,16.026336,582.181386,24.949139,53.164294


In [54]:
df.median(numeric_only=True)

id_mesure           1252.00
temperature           24.00
humidite              57.55
co2                  787.50
occupation            46.00
consommation_kwh     169.80
dtype: float64

**Réponse.** `df.describe()` fournit en une fois, pour chaque variable numérique :
`count` (valeurs non manquantes), `mean` (moyenne), `std` (écart-type), `min`, `max`,
et `25% / 50% / 75%` (les quartiles Q1, Q2, Q3). La **médiane** est la ligne `50%`,
confirmée par `df.median()` (ex. `temperature` : 24.00 dans les deux).

Variables potentiellement problématiques repérées dès les statistiques :
- `humidite` : min = −12 et max = 160 → **impossibles** (un % est entre 0 et 100) → incohérences (tâche 12) ;
- `temperature` : min = −30 et max = 96 → valeurs extrêmes suspectes → aberrantes probables (tâches 12, 15-16) ;
- `co2` : médiane ≈ 787 mais une valeur à 3900 → aberrante à examiner (tâche 16).

### 11) Y a-t-il des variables potentiellement problématiques ?

In [55]:
df.isna().sum()

id_mesure              0
date                   5
batiment               0
type_batiment          4
zone                   0
temperature           12
humidite              11
co2                    7
occupation             6
consommation_kwh       5
mode_climatisation     5
etat_systeme           0
jour_semaine           5
alerte                 0
dtype: int64

**Réponse.** Oui, l'exploration révèle plusieurs variables problématiques, de natures différentes :

- **Incohérences** : `humidite` (valeurs < 0 et > 100, impossibles pour un %), `date` (dates impossibles, déjà mises en `NaT` à la tâche 8).
- **Valeurs aberrantes** : `temperature` (min −30, max 96), `co2` (valeur à 3900 vs médiane ≈ 787).
- **Valeurs manquantes** : plusieurs colonnes (voir `df.isna().sum()`), à quantifier à la tâche 13.
- **Doublons potentiels** : `id_mesure` a 500 valeurs uniques pour 507 lignes → à examiner à la tâche 14.

Ces problèmes seront traités dans les tâches suivantes (12 : incohérences, 13 : manquantes, 14 : doublons, 15-16 : aberrantes).

### 12) Données incohérentes

#### 12a) Rechercher les valeurs d'humidité < 0

In [57]:
df[df["humidite"] < 0 ]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
281,1126,2025-02-01 06:00:00,B2,École,D,24.8,-5.0,759.0,57.0,225.2,Boost,Normal,Samedi,Non
335,1036,2025-01-09 18:00:00,B4,Bureau,D,21.2,-8.0,160.0,25.0,120.2,Normal,Alerte,Jeudi,Non
366,1216,2025-02-23 18:00:00,B3,Hôpital,B,23.2,-12.0,702.0,41.0,119.8,Normal,Normal,Dimanche,Non


In [58]:
(df["humidite"] < 0).sum()

np.int64(3)

#### 12b) Rechercher les valeurs d'humidité > 100

In [59]:
df[df["humidite"] > 100]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
59,1246,2025-03-03 06:00:00,B5,Centre commercial,A,24.2,108.0,813.0,18.0,-15.0,Normal,Normal,Lundi,Non
103,1016,2025-01-04 18:00:00,B6,Université,B,26.9,145.0,670.0,28.0,175.9,Normal,Normal,Samedi,Non
128,1156,2025-02-08 18:00:00,B1,Bureau,A,NaN,160.0,941.0,37.0,149.8,Eco,Normal,Samedi,Oui
160,1186,2025-02-16 06:00:00,B8,Entrepôt,B,24.9,125.0,NaN,18.0,123.5,Normal,Normal,Dimanche,Non
268,1276,2025-03-10 18:00:00,B1,Bureau,D,26.0,140.0,633.0,25.0,148.3,Normal,Normal,Lundi,Non
327,1066,2025-01-17 06:00:00,B2,École,C,25.8,132.0,646.0,14.0,93.6,Eco,Normal,Vendredi,Non
342,1096,2025-01-24 18:00:00,B8,Entrepôt,A,20.3,110.0,783.0,55.0,121.0,Normal,Normal,Vendredi,Non


In [60]:
(df["humidite"] > 100).sum()

np.int64(7)